### Install

In [ ]:
%%capture
!pip install torch torchvision langchain langgraph langchain-core \
             langchain-groq matplotlib scikit-learn gradio -q
print("✅ Done")

### Config

In [ ]:
import os
from google.colab import userdata

try:
    os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')
    print("✅ Groq key loaded")
except:
    print("⚠️ Add GROQ_API_KEY to Colab Secrets")

### Dataset

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

train_data = datasets.FashionMNIST(root="./data", train=True,  download=True, transform=transform)
test_data  = datasets.FashionMNIST(root="./data", train=False, download=True, transform=transform)

train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True)
test_loader  = DataLoader(test_data,  batch_size=BATCH_SIZE, shuffle=False)

CLASS_NAMES = ['T-shirt','Trouser','Pullover','Dress','Coat',
               'Sandal','Shirt','Sneaker','Bag','Ankle boot']

print(f"✅ FashionMNIST loaded")
print(f"   Train: {len(train_data)} | Test: {len(test_data)} | Classes: {len(CLASS_NAMES)}")

### Dynamic Model Builder

In [ ]:
import torch.nn as nn

ACTIVATION_MAP = {
    "relu":  nn.ReLU(),
    "leaky": nn.LeakyReLU(0.1),
    "gelu":  nn.GELU(),
    "tanh":  nn.Tanh(),
    "silu":  nn.SiLU()
}

OPTIMIZER_MAP = {
    "adam":    torch.optim.Adam,
    "sgd":     torch.optim.SGD,
    "adamw":   torch.optim.AdamW,
    "rmsprop": torch.optim.RMSprop
}

def build_model(config: dict) -> nn.Module:
    layers       = config.get("layers", [64, 128])
    activation   = ACTIVATION_MAP.get(config.get("activation", "relu"), nn.ReLU())
    dropout      = config.get("dropout", 0.3)
    use_batchnorm= config.get("batchnorm", True)

    blocks = []

    blocks += [nn.Conv2d(1, layers[0], kernel_size=3, padding=1),
               nn.BatchNorm2d(layers[0]) if use_batchnorm else nn.Identity(),
               activation, nn.MaxPool2d(2)]

    blocks += [nn.Conv2d(layers[0], layers[1], kernel_size=3, padding=1),
               nn.BatchNorm2d(layers[1]) if use_batchnorm else nn.Identity(),
               activation, nn.MaxPool2d(2)]

    if len(layers) > 2:
        blocks += [nn.Conv2d(layers[1], layers[2], kernel_size=3, padding=1),
                   nn.BatchNorm2d(layers[2]) if use_batchnorm else nn.Identity(),
                   activation]

    blocks += [nn.Flatten()]

    fc_in     = layers[-1] * 7 * 7
    fc_hidden = config.get("fc_hidden", 256)
    blocks   += [nn.Linear(fc_in, fc_hidden), activation,
                 nn.Dropout(dropout), nn.Linear(fc_hidden, 10)]

    return nn.Sequential(*blocks).to(DEVICE)

baseline_config = {
    "layers": [32, 64], "activation": "relu", "dropout": 0.3,
    "batchnorm": True, "fc_hidden": 128, "optimizer": "adam", "lr": 0.001
}

test_model = build_model(baseline_config)
print(f"✅ Model builder working | Params: {sum(p.numel() for p in test_model.parameters()):,}")

### Training Engine

In [ ]:
import numpy as np
from sklearn.metrics import f1_score, confusion_matrix

def train_model(config: dict, epochs: int = EPOCHS, verbose: bool = True):
    model        = build_model(config)
    optimizer_cls= OPTIMIZER_MAP.get(config.get("optimizer", "adam"), torch.optim.Adam)
    optimizer    = optimizer_cls(
        model.parameters(), lr=config.get("lr", 0.001),
        **({"momentum": 0.9} if config.get("optimizer") == "sgd" else {})
    )
    criterion  = nn.CrossEntropyLoss()
    scheduler  = torch.optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)
    history    = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}

    for epoch in range(epochs):
        model.train()
        t_loss, t_correct, t_total = 0, 0, 0
        for X, y in train_loader:
            X, y = X.to(DEVICE), y.to(DEVICE)
            optimizer.zero_grad()
            out  = model(X)
            loss = criterion(out, y)
            loss.backward()
            optimizer.step()
            t_loss    += loss.item()
            t_correct += (out.argmax(1) == y).sum().item()
            t_total   += y.size(0)

        model.eval()
        v_loss, v_correct, v_total = 0, 0, 0
        all_preds, all_labels = [], []
        with torch.no_grad():
            for X, y in test_loader:
                X, y = X.to(DEVICE), y.to(DEVICE)
                out  = model(X)
                loss = criterion(out, y)
                v_loss    += loss.item()
                v_correct += (out.argmax(1) == y).sum().item()
                v_total   += y.size(0)
                all_preds.extend(out.argmax(1).cpu().numpy())
                all_labels.extend(y.cpu().numpy())

        scheduler.step()

        train_acc  = t_correct / t_total
        val_acc    = v_correct / v_total
        train_loss = t_loss / len(train_loader)
        val_loss   = v_loss / len(test_loader)

        history["train_loss"].append(round(train_loss, 4))
        history["train_acc"].append(round(train_acc,   4))
        history["val_loss"].append(round(val_loss,     4))
        history["val_acc"].append(round(val_acc,       4))

        if verbose and (epoch + 1) % 2 == 0:
            print(f"  Epoch {epoch+1:02d}/{epochs} | "
                  f"Train {train_acc:.3f} / {train_loss:.4f} | "
                  f"Val {val_acc:.3f} / {val_loss:.4f}")

    f1  = round(f1_score(all_labels, all_preds, average="weighted"), 4)
    cm  = confusion_matrix(all_labels, all_preds)

    return {
        "model":           model,
        "history":         history,
        "final_val_acc":   round(val_acc,   4),
        "final_train_acc": round(train_acc, 4),
        "final_val_loss":  round(val_loss,  4),
        "f1_score":        f1,
        "overfit_gap":     round(train_acc - val_acc, 4),
        "confusion_matrix":cm.tolist(),
        "config":          config
    }

print("✅ Training engine ready")

### LangGraph Mutation Agent

In [ ]:
!pip install langchain-openai -q

In [ ]:
from langgraph.graph import StateGraph, END
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage
from typing import TypedDict, List, Dict
import json, copy

llm = ChatGroq(model=LLM_MODEL, temperature=0.2)

class MutationState(TypedDict):
    current_config:  Dict
    experiment_log:  List[Dict]
    best_result:     Dict
    mutation_count:  int
    done:            bool

def train_node(state: MutationState) -> MutationState:
    config = state["current_config"]
    run    = state["mutation_count"] + 1
    print(f"\n{'='*55}")
    print(f"🔬 Run {run}/{MAX_MUTATIONS} | Config: {config}")
    print(f"{'='*55}")

    result = train_model(config, epochs=EPOCHS)

    print(f"\n  ✅ Val Acc: {result['final_val_acc']:.4f} | "
          f"F1: {result['f1_score']:.4f} | "
          f"Overfit gap: {result['overfit_gap']:.4f}")

    log_entry = {
        "run":              run,
        "config":           copy.deepcopy(config),
        "final_val_acc":    result["final_val_acc"],
        "final_train_acc":  result["final_train_acc"],
        "final_val_loss":   result["final_val_loss"],
        "f1_score":         result["f1_score"],
        "overfit_gap":      result["overfit_gap"],
        "confusion_matrix": result["confusion_matrix"],
        "history":          result["history"]
    }

    best = state["best_result"]
    if not best or result["final_val_acc"] > best.get("final_val_acc", 0):
        best = copy.deepcopy(log_entry)
        best["model"] = result["model"]
        print(f"  🏆 New best model!")

    return {
        **state,
        "experiment_log": state["experiment_log"] + [log_entry],
        "best_result":    best,
        "mutation_count": run
    }

def mutate_node(state: MutationState) -> MutationState:
    if state["mutation_count"] >= MAX_MUTATIONS:
        return {**state, "done": True}

    last = state["experiment_log"][-1]

    history_summary = {
        "run":            last["run"],
        "val_acc":        last["final_val_acc"],
        "train_acc":      last["final_train_acc"],
        "val_loss":       last["final_val_loss"],
        "f1_score":       last["f1_score"],
        "overfit_gap":    last["overfit_gap"],
        "last_5_val_acc": last["history"]["val_acc"][-5:]
    }

    all_runs = [
        {"run": e["run"], "val_acc": e["final_val_acc"],
         "config": e["config"], "overfit_gap": e["overfit_gap"]}
        for e in state["experiment_log"]
    ]

    prompt = f"""You are an expert ML engineer acting as a neural architecture search agent.

CURRENT CONFIG:
{json.dumps(last["config"], indent=2)}

LATEST TRAINING RESULTS:
{json.dumps(history_summary, indent=2)}

ALL RUNS SO FAR:
{json.dumps(all_runs, indent=2)}

STRICT RULES FOR NEW CONFIG:
- layers MUST be a list of exactly 2 or 3 integers, each between 16 and 256
- Valid examples: [32,64], [64,128], [64,128,256], [32,64,128]
- NEVER output a single integer or empty list for layers
- activation must be one of: relu, leaky, gelu, tanh, silu
- dropout must be a float between 0.1 and 0.5
- batchnorm must be true or false
- fc_hidden must be one of: 64, 128, 256, 512
- optimizer must be one of: adam, sgd, adamw, rmsprop
- lr must be one of: 0.0001, 0.0005, 0.001, 0.005

DECISION RULES:
- If overfitting (overfit_gap > 0.05): increase dropout, reduce fc_hidden
- If underfitting (val_acc < 0.80): increase layers, try gelu or silu
- If plateauing: try different optimizer or learning rate
- If loss unstable: ensure batchnorm true, reduce lr

Respond ONLY in valid JSON with no markdown, no backticks, no extra text:
{{
  "reasoning": "2-3 sentences explaining what you observed and why",
  "mutation": "one sentence describing the specific change",
  "new_config": {{
    "layers": [int, int],
    "activation": "...",
    "dropout": float,
    "batchnorm": bool,
    "fc_hidden": int,
    "optimizer": "...",
    "lr": float
  }}
}}"""

    response = llm.invoke([HumanMessage(content=prompt)])
    try:
        clean = response.content.strip()
        if "```" in clean:
            clean = clean.split("```")[1].replace("json", "").strip()
        decision   = json.loads(clean)
        new_config = decision["new_config"]

        if not isinstance(new_config.get("layers"), list) or len(new_config["layers"]) < 2:
            raise ValueError("Invalid layers")

        print(f"\n🤖 Agent reasoning: {decision['reasoning']}")
        print(f"🔧 Mutation: {decision['mutation']}")

    except Exception as e:
        print(f"⚠️ Parse/validation error: {e} — applying safe fallback")
        new_config = {
            **last["config"],
            "dropout": min(0.5, round(last["config"].get("dropout", 0.3) + 0.05, 2)),
            "layers":  last["config"].get("layers", [32, 64])
        }

    return {**state, "current_config": new_config}

def should_continue(state: MutationState) -> str:
    return END if state["done"] else "train"

graph = StateGraph(MutationState)
graph.add_node("train",  train_node)
graph.add_node("mutate", mutate_node)
graph.set_entry_point("train")
graph.add_edge("train", "mutate")
graph.add_conditional_edges("mutate", should_continue, {END: END, "train": "train"})
agent = graph.compile()
print("✅ AutoMorph agent compiled")

### Run Agent

In [ ]:
# Force reload build_model with guard
import torch.nn as nn

ACTIVATION_MAP = {
    "relu":  nn.ReLU(),
    "leaky": nn.LeakyReLU(0.1),
    "gelu":  nn.GELU(),
    "tanh":  nn.Tanh(),
    "silu":  nn.SiLU()
}

OPTIMIZER_MAP = {
    "adam":    torch.optim.Adam,
    "sgd":     torch.optim.SGD,
    "adamw":   torch.optim.AdamW,
    "rmsprop": torch.optim.RMSprop
}

def build_model(config: dict) -> nn.Module:
    layers = config.get("layers", [64, 128])
    if not isinstance(layers, list) or len(layers) < 2:
        layers = [32, 64]
    layers = [max(8, l) for l in layers]

    activation    = ACTIVATION_MAP.get(config.get("activation", "relu"), nn.ReLU())
    dropout       = config.get("dropout", 0.3)
    use_batchnorm = config.get("batchnorm", True)

    blocks = []
    blocks += [nn.Conv2d(1, layers[0], kernel_size=3, padding=1),
               nn.BatchNorm2d(layers[0]) if use_batchnorm else nn.Identity(),
               activation, nn.MaxPool2d(2)]
    blocks += [nn.Conv2d(layers[0], layers[1], kernel_size=3, padding=1),
               nn.BatchNorm2d(layers[1]) if use_batchnorm else nn.Identity(),
               activation, nn.MaxPool2d(2)]
    if len(layers) > 2:
        blocks += [nn.Conv2d(layers[1], layers[2], kernel_size=3, padding=1),
                   nn.BatchNorm2d(layers[2]) if use_batchnorm else nn.Identity(),
                   activation]

    blocks += [nn.Flatten()]
    fc_in     = layers[-1] * 7 * 7
    fc_hidden = config.get("fc_hidden", 256)
    blocks   += [nn.Linear(fc_in, fc_hidden), activation,
                 nn.Dropout(dropout), nn.Linear(fc_hidden, 10)]

    return nn.Sequential(*blocks).to(DEVICE)

print("✅ build_model reloaded with guard")

In [ ]:
import traceback

print("🚀 AutoMorph Agent starting...\n")

try:
    final_state = agent.invoke({
        "current_config":  baseline_config,
        "experiment_log":  [],
        "best_result":     {},
        "mutation_count":  0,
        "done":            False
    })
    EXPERIMENT_LOG = final_state["experiment_log"]
    BEST_RESULT    = final_state["best_result"]
    print(f"\n✅ Complete — {len(EXPERIMENT_LOG)} runs")
    print(f"🏆 Best val acc : {BEST_RESULT['final_val_acc']:.4f}")
    print(f"🏆 Best F1      : {BEST_RESULT['f1_score']:.4f}")
    print(f"🏆 Best config  : {BEST_RESULT['config']}")
except Exception as e:
    traceback.print_exc()

In [ ]:
print("🚀 AutoMorph Agent starting...\n")

initial_state: MutationState = {
    "current_config":  baseline_config,
    "experiment_log":  [],
    "best_result":     {},
    "mutation_count":  0,
    "done":            False
}

final_state = agent.invoke(initial_state)

EXPERIMENT_LOG = final_state["experiment_log"]
BEST_RESULT    = final_state["best_result"]

print(f"\n{'='*55}")
print(f"✅ AutoMorph complete — {len(EXPERIMENT_LOG)} runs")
print(f"🏆 Best val accuracy : {BEST_RESULT['final_val_acc']:.4f}")
print(f"🏆 Best F1 score     : {BEST_RESULT['f1_score']:.4f}")
print(f"🏆 Best config       : {BEST_RESULT['config']}")

### Visualize Evolution

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

runs       = [e["run"]           for e in EXPERIMENT_LOG]
val_accs   = [e["final_val_acc"] for e in EXPERIMENT_LOG]
train_accs = [e["final_train_acc"]for e in EXPERIMENT_LOG]
f1_scores  = [e["f1_score"]      for e in EXPERIMENT_LOG]
overfit    = [e["overfit_gap"]   for e in EXPERIMENT_LOG]

fig = plt.figure(figsize=(16, 10))
fig.patch.set_facecolor("#0d1117")
gs  = gridspec.GridSpec(2, 2, figure=fig, hspace=0.4, wspace=0.35)

def styled_ax(ax, title):
    ax.set_facecolor("#161b22")
    ax.set_title(title, color="white", fontsize=11, pad=10)
    ax.tick_params(colors="white")
    for spine in ax.spines.values():
        spine.set_edgecolor("#30363d")
    ax.xaxis.label.set_color("white")
    ax.yaxis.label.set_color("white")

# Plot 1 — Accuracy evolution
ax1 = fig.add_subplot(gs[0, 0])
ax1.plot(runs, val_accs,   "o-", color="#7F77DD", linewidth=2, label="Val Acc",   markersize=7)
ax1.plot(runs, train_accs, "s--",color="#1D9E75", linewidth=2, label="Train Acc", markersize=5)
best_run = val_accs.index(max(val_accs)) + 1
ax1.axvline(best_run, color="#E24B4A", linestyle=":", alpha=0.7, label=f"Best run {best_run}")
ax1.legend(facecolor="#161b22", labelcolor="white", fontsize=8)
ax1.set_xlabel("Run"); ax1.set_ylabel("Accuracy")
styled_ax(ax1, "Accuracy Evolution Across Mutations")

# Plot 2 — F1 Score
ax2 = fig.add_subplot(gs[0, 1])
bars = ax2.bar(runs, f1_scores, color=["#EF9F27" if f == max(f1_scores) else "#378ADD" for f in f1_scores])
ax2.set_xlabel("Run"); ax2.set_ylabel("F1 Score")
styled_ax(ax2, "F1 Score Per Run")
for bar, val in zip(bars, f1_scores):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
             f"{val:.3f}", ha="center", color="white", fontsize=8)

# Plot 3 — Overfit gap
ax3 = fig.add_subplot(gs[1, 0])
colors = ["#E24B4A" if o > 0.05 else "#1D9E75" for o in overfit]
ax3.bar(runs, overfit, color=colors)
ax3.axhline(0.05, color="#EF9F27", linestyle="--", alpha=0.7, label="Overfit threshold")
ax3.legend(facecolor="#161b22", labelcolor="white", fontsize=8)
ax3.set_xlabel("Run"); ax3.set_ylabel("Train - Val Acc")
styled_ax(ax3, "Overfitting Gap Per Run")

# Plot 4 — Loss curves of best run
best_idx = val_accs.index(max(val_accs))
best_history = EXPERIMENT_LOG[best_idx]["history"]
ax4 = fig.add_subplot(gs[1, 1])
ep = range(1, len(best_history["val_loss"]) + 1)
ax4.plot(ep, best_history["train_loss"], "o-", color="#1D9E75", linewidth=2, label="Train loss")
ax4.plot(ep, best_history["val_loss"],   "s-", color="#7F77DD", linewidth=2, label="Val loss")
ax4.legend(facecolor="#161b22", labelcolor="white", fontsize=8)
ax4.set_xlabel("Epoch"); ax4.set_ylabel("Loss")
styled_ax(ax4, f"Loss Curves — Best Run (Run {best_idx+1})")

plt.suptitle("AutoMorph — Neural Architecture Mutation Agent", color="white", fontsize=14, y=1.01)
plt.savefig("automorph_results.png", dpi=150, bbox_inches="tight", facecolor="#0d1117")
plt.show()
print("✅ Saved automorph_results.png")

### Confusion Matrix

In [ ]:
# Check what keys BEST_RESULT has
print(BEST_RESULT.keys())

In [ ]:
import itertools

cm      = np.array(BEST_RESULT["confusion_matrix"])
cm_norm = cm.astype("float") / cm.sum(axis=1, keepdims=True)

fig, ax = plt.subplots(figsize=(12, 10))
fig.patch.set_facecolor("#0d1117")
ax.set_facecolor("#0d1117")

im = ax.imshow(cm_norm, interpolation="nearest", cmap=plt.cm.Blues)
plt.colorbar(im, ax=ax).ax.yaxis.set_tick_params(color="white")

ax.set_xticks(range(10)); ax.set_yticks(range(10))
ax.set_xticklabels(CLASS_NAMES, rotation=45, ha="right", color="white", fontsize=9)
ax.set_yticklabels(CLASS_NAMES, color="white", fontsize=9)

thresh = cm_norm.max() / 2
for i, j in itertools.product(range(10), range(10)):
    ax.text(j, i, f"{cm_norm[i,j]:.2f}", ha="center", va="center",
            color="white" if cm_norm[i,j] < thresh else "#0d1117", fontsize=7)

ax.set_title(f"Confusion Matrix — Best Model (Val Acc: {BEST_RESULT['final_val_acc']:.4f})",
             color="white", fontsize=12, pad=12)
ax.set_xlabel("Predicted", color="white")
ax.set_ylabel("True", color="white")
ax.tick_params(colors="white")
for spine in ax.spines.values():
    spine.set_edgecolor("#30363d")

plt.tight_layout()
plt.savefig("confusion_matrix.png", dpi=150, bbox_inches="tight", facecolor="#0d1117")
plt.show()
print("✅ Saved confusion_matrix.png")

### Experiment Log

In [ ]:
print("\n📋 FULL EXPERIMENT LOG")
print("="*75)
for e in EXPERIMENT_LOG:
    cfg = e["config"]
    print(f"\nRun {e['run']:02d} | Val: {e['final_val_acc']:.4f} | "
          f"F1: {e['f1_score']:.4f} | Overfit: {e['overfit_gap']:.4f}")
    print(f"  layers:{cfg['layers']} | act:{cfg['activation']} | "
          f"drop:{cfg['dropout']} | opt:{cfg['optimizer']} | lr:{cfg['lr']}")

print(f"\n{'='*75}")
print(f"🏆 BEST VAL ACC : {BEST_RESULT['final_val_acc']:.4f}")
print(f"🏆 BEST F1      : {BEST_RESULT['f1_score']:.4f}")
print(f"🏆 BEST CONFIG  : {BEST_RESULT['config']}")

### Gradio Dashboard

In [ ]:
import gradio as gr

def overview():
    lines = [
        "## 🧬 AutoMorph — Experiment Summary\n",
        f"**Total runs:** {len(EXPERIMENT_LOG)}  |  "
        f"**Best val accuracy:** {BEST_RESULT['final_val_acc']:.4f}  |  "
        f"**Best F1:** {BEST_RESULT['f1_score']:.4f}\n",
        f"**Best config:** `{BEST_RESULT['config']}`"
    ]
    return "\n".join(lines)

def run_detail(run_num):
    try:
        e = EXPERIMENT_LOG[int(run_num) - 1]
        return f"""## Run {e['run']}
**Val accuracy:** {e['final_val_acc']:.4f} | **F1:** {e['f1_score']:.4f} | **Overfit gap:** {e['overfit_gap']:.4f}

**Config:**
```
{json.dumps(e['config'], indent=2)}
```
**Val acc per epoch:** {e['history']['val_acc']}
**Train loss per epoch:** {e['history']['train_loss']}"""
    except:
        return f"Run {run_num} not found. Available: 1 to {len(EXPERIMENT_LOG)}"

def ask_agent(question):
    context = "\n".join([
        f"Run {e['run']}: val_acc={e['final_val_acc']}, f1={e['f1_score']}, "
        f"overfit={e['overfit_gap']}, config={e['config']}"
        for e in EXPERIMENT_LOG
    ])
    response = llm.invoke([HumanMessage(content=
        f"You are AutoMorph, an ML experiment analysis agent.\n\n"
        f"EXPERIMENT LOG:\n{context}\n\n"
        f"BEST RESULT: {BEST_RESULT['config']} — val_acc={BEST_RESULT['final_val_acc']}\n\n"
        f"QUESTION: {question}\n\nAnswer concisely with specific run references."
    )])
    return response.content

with gr.Blocks(title="AutoMorph", theme=gr.themes.Base()) as demo:
    gr.Markdown("# 🧬 AutoMorph — Autonomous Neural Architecture Mutation Agent")
    with gr.Tabs():
        with gr.Tab("📊 Overview"):
            gr.Markdown(overview())
            with gr.Row():
                gr.Image("automorph_results.png", label="Evolution Charts")
                gr.Image("confusion_matrix.png",  label="Confusion Matrix")
        with gr.Tab("🔬 Run Explorer"):
            inp = gr.Textbox(label="Run number", placeholder="1")
            out = gr.Markdown()
            gr.Button("Inspect").click(run_detail, inp, out)
        with gr.Tab("🤖 Ask Agent"):
            q = gr.Textbox(label="Question",
                           placeholder="Why did run 3 overfit? What was the best activation?")
            a = gr.Markdown()
            gr.Button("Ask ↗").click(ask_agent, q, a)

demo.launch(share=True)